# Step 11: Conditional Workflows

        _Instructor Solution Notebook_  
        Estimated time: **75 minutes**  
        Difficulty: **Intermediate-Advanced**

        ## Learning Objectives
        - [ ] Route messages based on conditions.
- [ ] Model branching logic with small executors.
- [ ] Choose specialized paths for technical and business inputs.
- [ ] Keep fallback behavior explicit.

        ## Prerequisites
        - Step 10 completed.


## Table of Contents

1. Setup & Imports
2. Part 1: Introduction
3. Part 2: Core Implementation
4. Part 3: Hands-On Exercises
5. Part 4: Solutions & Discussion
6. Part 5: Best Practices & Tips
7. Summary & Key Takeaways
8. Additional Resources


## Setup & Imports

The next cell adds the repository root to `sys.path` and imports the shared course helpers.
Most notebooks use the same bootstrap so the lesson content can stay focused on the concept,
not on path setup.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "helpers").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from helpers import (
    LocalTfidfVectorStore,
    SQLiteConversationMemory,
    assert_minimum_python,
    chunk_documents,
    create_agent,
    create_chat_client,
    describe_response,
    load_settings,
    load_text_documents,
    print_banner,
    print_json,
    resolve_notebook_root,
    summarize_session,
    validate_provider_configuration,
)


## Part 1: Introduction

Once dataflow becomes non-linear, the workflow needs explicit routing rules. This notebook shows a minimal branching setup without hiding the logic behind a black box.

Expected output and validation notes:

Expected output snapshot:

- Technical input yields a `TECH PATH` result
- Business input yields a `BUSINESS PATH` result


## Part 2: Core Implementation

### Route to specialized executors


In [ ]:
from typing_extensions import Never
from agent_framework import Executor, WorkflowBuilder, WorkflowContext, handler

class Router(Executor):
    @handler
    async def process(self, text: str, ctx: WorkflowContext[dict]) -> None:
        route = "technical" if "python" in text.lower() or "code" in text.lower() else "business"
        await ctx.send_message({"route": route, "text": text})

class TechnicalExecutor(Executor):
    @handler
    async def process(self, payload: dict, ctx: WorkflowContext[Never, str]) -> None:
        await ctx.yield_output(f"TECH PATH: {payload['text']}")

class BusinessExecutor(Executor):
    @handler
    async def process(self, payload: dict, ctx: WorkflowContext[Never, str]) -> None:
        await ctx.yield_output(f"BUSINESS PATH: {payload['text']}")

router = Router(id="router")
technical = TechnicalExecutor(id="technical")
business = BusinessExecutor(id="business")

conditional = (
    WorkflowBuilder(start_executor=router)
    .add_edge(router, technical, condition=lambda payload: payload["route"] == "technical")
    .add_edge(router, business, condition=lambda payload: payload["route"] == "business")
    .build()
)


## Part 2: Core Implementation

### Try both branches


In [ ]:
print((await conditional.run("How do I write async Python code?")).get_outputs())
print((await conditional.run("How should we position this training program for managers?")).get_outputs())


## Part 3: Hands-On Exercises

Add a third fallback path for general questions that do not match either primary route.

This mirrored notebook uses completed exercise code so instructors can demonstrate the target state.


In [ ]:
class GeneralExecutor(Executor):
    @handler
    async def process(self, payload: dict, ctx: WorkflowContext[Never, str]) -> None:
        await ctx.yield_output(f"GENERAL PATH: {payload['text']}")

general = GeneralExecutor(id="general")

def classify_route(payload: dict) -> str:
    text = payload["text"].lower()
    if "python" in text or "code" in text:
        return "technical"
    if "strategy" in text or "program" in text or "manager" in text:
        return "business"
    return "general"

class RouterV2(Executor):
    @handler
    async def process(self, text: str, ctx: WorkflowContext[dict]) -> None:
        await ctx.send_message({"route": classify_route({"text": text}), "text": text})

router_v2 = RouterV2(id="router_v2")
conditional_v2 = (
    WorkflowBuilder(start_executor=router_v2)
    .add_edge(router_v2, technical, condition=lambda payload: payload["route"] == "technical")
    .add_edge(router_v2, business, condition=lambda payload: payload["route"] == "business")
    .add_edge(router_v2, general, condition=lambda payload: payload["route"] == "general")
    .build()
)

print((await conditional_v2.run("How do I structure Python async tools?")).get_outputs())
print((await conditional_v2.run("How should we brief managers on this program?")).get_outputs())
print((await conditional_v2.run("Summarize the course in one sentence.")).get_outputs())


## Part 4: Solutions & Discussion

Use this section to compare your implementation with one clear working approach. The goal is not
perfect matching output; the goal is a sound pattern that is easy to explain, debug, and extend.


In [ ]:
class GeneralExecutor(Executor):
    @handler
    async def process(self, payload: dict, ctx: WorkflowContext[Never, str]) -> None:
        await ctx.yield_output(f"GENERAL PATH: {payload['text']}")

general = GeneralExecutor(id="general")

def classify_route(payload: dict) -> str:
    text = payload["text"].lower()
    if "python" in text or "code" in text:
        return "technical"
    if "strategy" in text or "program" in text or "manager" in text:
        return "business"
    return "general"

class RouterV2(Executor):
    @handler
    async def process(self, text: str, ctx: WorkflowContext[dict]) -> None:
        await ctx.send_message({"route": classify_route({"text": text}), "text": text})

router_v2 = RouterV2(id="router_v2")
conditional_v2 = (
    WorkflowBuilder(start_executor=router_v2)
    .add_edge(router_v2, technical, condition=lambda payload: payload["route"] == "technical")
    .add_edge(router_v2, business, condition=lambda payload: payload["route"] == "business")
    .add_edge(router_v2, general, condition=lambda payload: payload["route"] == "general")
    .build()
)

print((await conditional_v2.run("How do I structure Python async tools?")).get_outputs())
print((await conditional_v2.run("How should we brief managers on this program?")).get_outputs())
print((await conditional_v2.run("Summarize the course in one sentence.")).get_outputs())


## Part 5: Best Practices & Tips

        - Keep classification logic inspectable and testable.
- Add fallback routes deliberately instead of letting messages disappear.
- Treat routing rules as product logic, not just plumbing.


## Summary & Key Takeaways

You now have a concrete branching workflow that routes messages to specialized handlers.


## Additional Resources

        - `Step 10 notebook`
- `agent_framework WorkflowBuilder conditional edges`
